In [ ]:
import os
from pathlib import Path

os.environ["NJ_DOMAIN"] = "v3"  # must precede the nj_sfincs import

import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nj_sfincs").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from nj_sfincs import animate, domain, plots

DOM = domain.active()
EXP = ROOT / "experiments" / DOM.name
FIGS = ROOT / "reports" / "figures"
TAG = "2026-09-14"

# The fixed-engine epoch (native v2.3.3 + nj-winddir-fix-1). Keys are the panel labels.
# Filtered below to the arms that have both a map and a scored metrics row, so this
# notebook re-executes cleanly as later arms land.
CANDIDATES = {
    "naccs-premier": "naccs-premier",
    "wave-noig": "wave-noig",
    "wave-fw02": "wave-fw02",
    "bed-nobuildings": "bed-nobuildings",
    "naccs-nowaves": "naccs-nowaves",
    "wave-apex": "wave-apex",
}
_m = pd.read_csv(EXP / "metrics.csv", index_col=0)
RUNS = {k: v for k, v in CANDIDATES.items()
        if (EXP / v / "sfincs_map.nc").exists() and v in _m.index}
NCOL = 3

print(DOM.name, "|", sorted(DOM.map_windows))
print("in this notebook:", list(RUNS))
print("not yet scored:", [k for k in CANDIDATES if k not in RUNS])


## Arms — every run in this notebook is on the fixed engine (`bin:v2.3.3-winddir-fix-1`, waves launched in the imposed direction); one row = one change from the premier

| arm | wind growth | `snapwave_fw` | IG waves | buildings in subgrid | SnapWave band | what it isolates |
|---|---|---|---|---|---|---|
| `naccs-premier` | on | 0.01 | on | yes | shelf-steps | the baseline |
| `wave-noig` | on | 0.01 | **off** | yes | shelf-steps | infragravity waves |
| `wave-fw02` | on | **0.02** | on | yes | shelf-steps | bottom friction (the old value) |
| `bed-nobuildings` | on | 0.01 | on | **no** | shelf-steps | the building-footprint tier |
| `naccs-nowaves` | – | – | – | yes | – | SnapWave altogether |
| `wave-apex` | on | 0.01 | on | yes | **extended north to Rockaway** | swell supply into Lower Bay |


## Water-level boundary — forced cells + NACCS support (every arm shares one boundary; one panel)

In [ ]:
plots.plot_waterlevel_boundary_panels({"naccs-premier": "naccs-premier"}, ncol=1);

## Metrics

In [ ]:
csv = EXP / "metrics.csv"
m = pd.read_csv(csv, index_col=0).loc[list(RUNS.values())]
HEADLINE = [
    "hwm_n_scored", "hwm_rmse_scored_m", "hwm_bias_scored_m",
    "hwm_within0.5_scored", "motf_csi", "motf_pod", "motf_far",
    "motf_far_connected", "motf_csi_connected", "motf_km2_excluded_boxes",
    "engine", "snapwave_direction", "subgrid",
]
display(m[[c for c in HEADLINE if c in m.columns]].round(3))


## Gauges — ⚠️ Ship Bottom · Sea Isle · Stone Harbor obs peaks are pre-storm gap artefacts; Sandy Hook died mid-storm (read its pre-failure peak)

In [ ]:
plots.plot_gauge_verification(RUNS, ncol=NCOL);

## Gauge metrics — `tide` gauges: read range/phase. `surge` gauges: read peak.

In [ ]:
m = pd.read_csv(EXP / "metrics.csv", index_col=0).loc[list(RUNS.values())]

COLS = {
    "peak_obs": "peak_obs_{n}_m",
    "peak_mod": "peak_mod_full_{n}_m",
    "peak_err": "peak_err_{n}_m",
    "peak_lag_min": "peak_lag_{n}_min",
    "tide_obs_rng": "tide_obs_range_{n}_m",
    "tide_mod_rng": "tide_mod_range_{n}_m",
    "tide_damping": "tide_range_damping_{n}_m",
    "phase_lag_min": "phase_lag_{n}_min",
}

rows = []
for arm in m.index:
    for g in DOM.obs_gauges:
        r = {"arm": arm, "gauge": g.name, "kind": g.kind,
             "crest": "survives" if g.survives_crest else "DIED"}
        for label, pat in COLS.items():
            k = pat.format(n=g.name)
            r[label] = m.loc[arm, k] if k in m.columns else float("nan")
        k = f"peak_err_prefail_{g.name}_m"
        if k in m.columns:
            r["peak_err"] = m.loc[arm, k]
            r["crest"] = "DIED (prefail)"
        rows.append(r)

gm = pd.DataFrame(rows).set_index(["gauge", "kind", "crest", "arm"]).round(3)
display(gm)


## HWM — ⚠️ bay marks carry ±0.3 m of seiche PHASE between arms (FINDINGS §40); compare arms PAIRED, never by pooled RMSE alone

In [ ]:
plots.plot_hwm_residual_panels(RUNS, ncol=NCOL);

## MOTF — `naccs-nowaves` is a legitimate configuration but its extent is not ranked against waves-on arms; buildings arms dry their footprints (compare on the masked CSI)

In [ ]:
plots.plot_motf_panels(RUNS, ncol=NCOL, split_fa=True);

## Animations — the premier

In [ ]:
anim = animate.animate_field("naccs-premier", "depth", window="raritan", fps=6)
p = FIGS / f"v3_depth_raritan_{TAG}.gif"
anim.save(p, writer="pillow", fps=6, dpi=90)
display(Image(filename=str(p)))


In [ ]:
anim = animate.animate_field("naccs-premier", "depth", window="cape_may", fps=6)
p = FIGS / f"v3_depth_cape_may_{TAG}.gif"
anim.save(p, writer="pillow", fps=6, dpi=90)
display(Image(filename=str(p)))


In [ ]:
anim = animate.animate_field("naccs-premier", "hm0", window="cape_may", fps=6)
p = FIGS / f"v3_hm0_cape_may_{TAG}.gif"
anim.save(p, writer="pillow", fps=6, dpi=90)
display(Image(filename=str(p)))


## Interactive — pick a run, field, window

In [ ]:
import holoviews as hv
import ipywidgets as widgets

hv.extension("bokeh")


@widgets.interact(
    run=list(RUNS.values()),
    var=["depth", "zs", "hm0", "tp"],
    window=sorted(DOM.map_windows),
)
def browse(run="naccs-premier", var="depth", window="absecon"):
    display(animate.explore_field(run, var=var, window=window))
